<a href="https://colab.research.google.com/github/sneyx123-github/CopilotStudioSamples/blob/master/DrStop_DnsMapping_v1.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The provided Python notebook efficiently manages Cloud Run
deployments by handling region compatibility, existing state errors, and the 24-hour lock restriction for SSL challenges. A pre-flight DNS check using Python's socket library is recommended to enhance the tool's plug-and-play capability by verifying IP matches before gcloud execution. You can find the sample notebook at GitHub.

---

https://stackoverflow.com/questions/62596466/how-can-i-run-notebooks-of-a-github-project-in-google-colab

---



In [ ]:
from google.colab import auth
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"

# Melde dich mit deinem Google-Konto an
auth.authenticate_user()

# Setze das Projekt für die gcloud CLI
!gcloud config set project {PROJECT_ID}




In [2]:
from google.colab import auth
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"

if 0:
  # Melde dich mit deinem Google-Konto an
  auth.authenticate_user()
else:
  !gcloud auth login

# Setze das Projekt für die gcloud CLI
!gcloud config set project {PROJECT_ID}


Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=n5XNSIC5vcOEewsXNoeMDt3oWvrRPL&prompt=consent&token_usage=remote&access_type=offline&code_challenge=w6vhbr8aUv8gPrzhBqwYrCRQRgQdwLDcOCHn82QqxPg&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E-lvd8ywIBRSMIExOnuMpSUlEauCZq7t7CpBbWKJEgvi7B0z-Oyc7MIs0X_6JzeVw

You are now logged in as [simonborisney@gmail.com].
Your current projec

In [ ]:
import time
from datetime import datetime

# WICHTIG: Nur die Subdomain verwenden, kein http:// oder ://
SERVICE = "gradio-int"
DOMAIN = f"{SERVICE}.us.kommunikator-gmbh.com"
REGION = "europe-west1"

if 1:
  # 1. Altes Mapping löschen
  print(f"🗑️ Lösche altes Mapping für {DOMAIN}...")
  !gcloud beta run domain-mappings delete --domain {DOMAIN} --region {REGION} --quiet

  time.sleep(10)

  # 2. Mapping neu erstellen
  print(f"🚀 Erstelle Mapping neu für {DOMAIN}...")
  !gcloud beta run domain-mappings create --service {SERVICE} --domain {DOMAIN} --region {REGION}

# 3. Überwachungsschleife
print(f"⏳ Überwachung gestartet für {DOMAIN}. Prüfe alle 5 Minuten...")

while True:
    status_output = !gcloud beta run domain-mappings describe --domain {DOMAIN} --region {REGION} --format="yaml(status.conditions)"
    status_str = "\n".join(status_output)
    now = datetime.now().strftime('%H:%M:%S')

    if "status: 'True'\n    type: Ready" in status_str:
        print(f"✅ [{now}] ERFOLG! Deine App ist LIVE unter https://{DOMAIN}")
        break
    elif "message: Certificate issuance pending" in status_str:
        print(f"🔄 [{now}] Google prüft DNS... (Certificate issuance pending)")
    elif "reason: CertificatePending" in status_str:
        print(f"⏳ [{now}] Warte auf Zertifikat-Erstellung...")
    else:
        print(f"📡 [{now}] Status: {status_str[:100]}...") # Zeigt den Anfang des Status an

    time.sleep(300)

🗑️ Lösche altes Mapping für gradio-int.us.kommunikator-gmbh.com...
Mappings to [gradio-int.us.kommunikator-gmbh.com] now have been deleted.
🚀 Erstelle Mapping neu für gradio-int.us.kommunikator-gmbh.com...
Waiting for certificate provisioning. You must configure your DNS records for certificate issuance to begin.
NAME        RECORD TYPE  CONTENTS
gradio-int  CNAME        ghs.googlehosted.com.
⏳ Überwachung gestartet für gradio-int.us.kommunikator-gmbh.com. Prüfe alle 5 Minuten...
⏳ [00:36:29] Warte auf Zertifikat-Erstellung...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"
DOMAIN_SERVICE = "gradio-service.us.kommunikator-gmbh.com"
REGION = "europe-west1"

print(f"Describing domain mapping for {DOMAIN_SERVICE}:")
!gcloud beta run domain-mappings describe --domain {DOMAIN_SERVICE} --region {REGION} --project {PROJECT_ID} --format="yaml"

Describing domain mapping for gradio-service.us.kommunikator-gmbh.com:
apiVersion: domains.cloudrun.com/v1
kind: DomainMapping
metadata:
  annotations:
    run.googleapis.com/operation-id: 7ebb026c-ed8f-44a8-a263-6bb2728f0895
    serving.knative.dev/creator: simonborisney@gmail.com
    serving.knative.dev/lastModifier: simonborisney@gmail.com
  creationTimestamp: '2026-04-06T17:14:44.656770Z'
  generation: 1
  labels:
    cloud.googleapis.com/location: europe-west1
    run.googleapis.com/overrideAt: '2026-04-06T17:14:48.996Z'
  name: gradio-service.us.kommunikator-gmbh.com
  namespace: '1074528386995'
  resourceVersion: AAZOzevKrj8
  selfLink: /apis/domains.cloudrun.com/v1/namespaces/1074528386995/domainmappings/gradio-service.us.kommunikator-gmbh.com
  uid: c1493488-5d39-496a-80dc-e65f44c5dde1
spec:
  routeName: gradio-service
status:
  conditions:
  - lastTransitionTime: '2026-04-06T17:23:11.454783Z'
    status: 'True'
    type: Ready
  - lastTransitionTime: '2026-04-06T17:23:11.4547

In [ ]:
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"
DOMAIN_TEST = "gradio-test.us.kommunikator-gmbh.com"
REGION = "europe-west1"

print(f"Describing domain mapping for {DOMAIN_TEST}:")
!gcloud beta run domain-mappings describe --domain {DOMAIN_TEST} --region {REGION} --project {PROJECT_ID} --format="yaml"

Describing domain mapping for gradio-test.us.kommunikator-gmbh.com:
apiVersion: domains.cloudrun.com/v1
kind: DomainMapping
metadata:
  annotations:
    run.googleapis.com/operation-id: 0284a35e-5e34-4080-b095-f2c65491d46d
    serving.knative.dev/creator: simonborisney@gmail.com
    serving.knative.dev/lastModifier: simonborisney@gmail.com
  creationTimestamp: '2026-04-06T19:25:08.433305Z'
  generation: 1
  labels:
    cloud.googleapis.com/location: europe-west1
    run.googleapis.com/overrideAt: '2026-04-06T19:25:12.128Z'
  name: gradio-test.us.kommunikator-gmbh.com
  namespace: '1074528386995'
  resourceVersion: AAZOz8yP9Nc
  selfLink: /apis/domains.cloudrun.com/v1/namespaces/1074528386995/domainmappings/gradio-test.us.kommunikator-gmbh.com
  uid: 0c69805d-76b4-484a-b55d-fc4eaded9f3b
spec:
  routeName: gradio-test
status:
  conditions:
  - lastTransitionTime: '2026-04-06T19:37:37.447127Z'
    status: 'True'
    type: Ready
  - lastTransitionTime: '2026-04-06T19:37:37.447127Z'
    sta